# 2s-AGCN: Two-Stream Adaptive Graph Conv Network on MediaPipe Pose Sequences

Orchestration notebook only -- all logic lives in `src/`. This is the protocol's
section-7 **primary target**, tried after the BiLSTM baseline validated the
pipeline (`notebooks/baseline/01_baseline_bilstm.ipynb`).

Why this over the BiLSTM: the BiLSTM had to learn *which joints even relate to
each other* from raw xyz, on only 529 training clips -- root cause of its
punching/shooting confusion (see baseline notebook's diagnosis). A GCN bakes the
skeleton graph in as fixed structure (`src.models.graph`, built from
`config.BONE_PAIRS`), so it only has to learn motion patterns on top of known
topology instead of discovering topology too -- more data-efficient per clip.

Simplifications vs. the original 2s-AGCN paper (see `src/models/agcn.py`
docstring for full rationale): single adaptive-adjacency subset instead of the
paper's 3-way partition, trained from scratch instead of fine-tuned from an
NTU-RGB+D checkpoint (25-joint layout doesn't line up with our joint subset
without a lossy remap, and pulls in mmcv/mmaction2 -- painful to build on
Windows). Two streams (joint positions + bone vectors) trained jointly, logits
summed.

**v2 note:** the first pass used a 14-joint body-only graph (elbow/knee/hip
skeleton) and landed on 82% test -- same punching<->shooting confusion the
BiLSTM had (punch recall 0.60). Root cause: neither model could see fist vs.
open-palm shape. The graph now includes 6 fingertip nodes (`src.models.graph.GCN_JOINTS`,
20 nodes total) bone-connected to their wrist.

**v3 note:** added an explicit `HandOpennessStream` (wrist->fingertip
distance, undiluted classifier vote) on top of v2's 20-node graph -- test
jumped to 97.96%, punch recall 1.0. But live-camera testing (not the locked
test clips) showed shooting still unreliable: Pose's own fingertip landmarks
(17-22) run 0.71-0.78 mean visibility and dip as low as 0.06 *even on the
curated recording setup* -- the exact signal v3 leaned on hardest (`hand_w`
learned to ~2x) is also the least trustworthy one live.

**v4 (current): real finger landmarks.** Replaced Pose's crude fingertip
proxy with a dedicated MediaPipe Hands model (`src.data.extract_hands`, 21
landmarks/hand) run over the same clip videos, reduced to a per-finger
curl-angle signal (`src.data.hand_features`, fist=0, extended=1) that's
rotation-invariant and doesn't depend on Pose's low-confidence tip
landmarks at all. Packed as 2 extra "pseudo-nodes" after the 20 physical
skeleton nodes (`src.models.graph.TOTAL_PACKED_NODES` = 22) -- see
`src.models.agcn.HandOpennessStream` v4 docstring.

Run order:
1. Load labels, confirm participant split (same locked split as the BiLSTM)
2. Build graph cache -- (SEQ_LEN, 22, 6) joint+bone+hand-curl tensors,
   reusing cached MediaPipe Pose landmarks (`data/interim/`) and extracting
   MediaPipe Hands landmarks fresh (`data/interim_hands/`)
3. Train `TwoStreamAGCN`
4. Evaluate on locked val/test split, with TTA at inference (addendum #6)

See `action_classifier_protocol.md` section 7 for the model-selection rationale.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent if (Path.cwd().name == "agcn") else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch

from src.config import (
    ACTIONS, PROCESSED_DIR, RUNS_DIR, SEED, SEQ_LEN, TEST_PARTICIPANTS,
    TRAIN_PARTICIPANTS, VAL_PARTICIPANTS,
)

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("project root:", PROJECT_ROOT)
print("torch:", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

project root: c:\Data_Tekken
torch: 2.5.1+cpu | device: cpu


## 1. Load labels + confirm split

Same locked participant split as the BiLSTM baseline -- no leakage, no re-deciding it here.

In [2]:
from src.data.dataset import load_labels, split_dataframe

df = load_labels()
train_df, val_df, test_df = split_dataframe(df)

print(f"total clips: {len(df)}")
print(f"train: {len(train_df)} clips (participants {TRAIN_PARTICIPANTS}, side+front)")
print(f"val:   {len(val_df)} clips (participant {VAL_PARTICIPANTS}, side-only)")
print(f"test:  {len(test_df)} clips (participants {TEST_PARTICIPANTS}, side-only)")
assert set(train_df.participant_id) & set(val_df.participant_id) == set()
assert set(train_df.participant_id) & set(test_df.participant_id) == set()
assert set(val_df.participant_id) & set(test_df.participant_id) == set()
print("\nno participant leakage across splits -- confirmed")

total clips: 642
train: 529 clips (participants [4, 5, 6, 7, 8, 9], side+front)
val:   30 clips (participant [1], side-only)
test:  49 clips (participants [2, 3], side-only)

no participant leakage across splits -- confirmed


## 2. Build graph cache

`src.data.graph_dataset.build_graph_cache` mirrors `build_feature_cache` (same
atomic-write/cache-validation, same `augment_clip` call, train-split-only
augmentation) but saves the structured `(SEQ_LEN, 22, 6)` joint+bone+hand-curl
tensor (`src.models.graph.joint_bone_from_sample`) instead of the flattened
279-dim BiLSTM vector. 22 nodes = 20 physical skeleton nodes (14-joint body +
6 fingertips) + 2 hand-curl pseudo-nodes fed by a fresh MediaPipe Hands pass
(`src.data.extract_hands`, cached under `data/interim_hands/`) instead of
Pose's low-confidence fingertip landmarks. Reuses the MediaPipe Pose landmarks
already cached in `data/interim/` -- only the Hands extraction is new work
here, so this cell is slower the first time it runs per clip.

In [3]:
from src.data.graph_dataset import build_graph_cache
from src.data.dataset import ClipSequenceDataset

train_paths = build_graph_cache(train_df, split="train", augment=True)
val_paths = build_graph_cache(val_df, split="val", augment=False)
test_paths = build_graph_cache(test_df, split="test", augment=False)

train_ds = ClipSequenceDataset(train_paths)
val_ds = ClipSequenceDataset(val_paths)
test_ds = ClipSequenceDataset(test_paths)

sample_feat, _ = train_ds[0]
print(f"train: {len(train_ds)} (incl. augmented) | val: {len(val_ds)} | test: {len(test_ds)}")
print(f"feat shape: {tuple(sample_feat.shape)}  (expected (SEQ_LEN=40, TOTAL_PACKED_NODES=22, 6))")
assert tuple(sample_feat.shape) == (SEQ_LEN, 22, 6), "shape mismatch -- stale cache? clear data/processed/graph_* and rerun"

train: 1587 (incl. augmented) | val: 30 | test: 49
feat shape: (40, 22, 6)  (expected (SEQ_LEN=40, TOTAL_PACKED_NODES=22, 6))


## 3. Train TwoStreamAGCN

Reuses `src.train.train_baseline` unmodified -- `TwoStreamAGCN.forward` returns
`(logits, aux)` matching `BiLSTMBaseline`'s `(logits, attn_weights)` contract,
so the training loop, checkpointing, and per-run `runs/<name>_<timestamp>/`
folder all just work.

In [ ]:
from src.models.agcn import TwoStreamAGCN
from src.train import train_baseline

model = TwoStreamAGCN(base_channels=32, num_classes=3, dropout=0.3)
n_params = sum(p.numel() for p in model.parameters())
print(f"params: {n_params:,}")

run_dir, best_val_acc = train_baseline(
    model, train_ds, val_ds, run_name="agcn_2stream", device=DEVICE,
    extra_config={"base_channels": 32, "total_packed_nodes": 22, "seq_len": SEQ_LEN, "variant": "v4_hands"},
)
print(f"\nbest val acc: {best_val_acc:.3f}  |  run saved to {run_dir}")
print(f"learned stream weights (joint, bone, hand): "
      f"{model.joint_w.item():.3f}, {model.bone_w.item():.3f}, {model.hand_w.item():.3f}")

params: 578,352


## 4. Evaluate on locked test set (deterministic + TTA)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

from src.evaluate import evaluate_dataset

model.load_state_dict(torch.load(run_dir / "best.pt", weights_only=True))
model.to(DEVICE)
model.eval()

report, cm = evaluate_dataset(model, test_ds, device=DEVICE, class_names=ACTIONS)
print("=== 2s-AGCN -- test (deterministic) ===")
print(pd.DataFrame(report).T)
print(cm)

=== 2s-AGCN -- test (deterministic) ===
              precision    recall  f1-score    support
kicking        1.000000  1.000000  1.000000  16.000000
punching       0.937500  1.000000  0.967742  15.000000
shooting       1.000000  0.944444  0.971429  18.000000
accuracy       0.979592  0.979592  0.979592   0.979592
macro avg      0.979167  0.981481  0.979724  49.000000
weighted avg   0.980867  0.979592  0.979629  49.000000
[[16  0  0]
 [ 0 15  0]
 [ 0  1 17]]


In [ ]:
from tqdm.auto import tqdm

from src.config import ACTION_TO_IDX
from src.data.dataset import _raw_landmarks
from src.data.graph_dataset import _raw_hand_landmarks, predict_with_tta_agcn

tta_preds, tta_labels = [], []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="TTA eval"):
    raw = _raw_landmarks(row)
    hand_raw = _raw_hand_landmarks(row)
    pred = predict_with_tta_agcn(model, raw["xyz"], raw["visibility"], hand_raw, device=DEVICE)
    tta_preds.append(pred)
    tta_labels.append(ACTION_TO_IDX[row["action"]])

tta_preds, tta_labels = np.array(tta_preds), np.array(tta_labels)
print("=== 2s-AGCN -- test (TTA: orig + mirror + 2x time-warp) ===")
print(classification_report(tta_labels, tta_preds, target_names=ACTIONS))
print(confusion_matrix(tta_labels, tta_preds))

## 5. Leave-participant-out CV -- follow-up, not run here

Same caveat as the BiLSTM notebook: a single held-out split (2-3 people) is a
noisy estimate with only 9 participants, but full group-k-fold CV means
retraining this model N times -- worth doing once the single-split number
looks promising enough to justify the GPU time, mirroring
`src.evaluate.bilstm_group_cv`'s pattern but swapping in `TwoStreamAGCN` +
`build_graph_cache`.

## 6. Summary

Fill in after running:

- 2s-AGCN v4 (real MediaPipe Hands curl signal) test accuracy (deterministic): ___
- 2s-AGCN v4 test accuracy (TTA): ___
- vs. v3 (Pose-fingertip-distance proxy, 97.96% test/TTA, punch recall 1.0
  on the locked test set but unreliable live per the notebook's v3 note): ___
- confusion matrix pattern -- does shooting hold up now on live camera too,
  not just the locked test clips?

**Next step:** live-test with `scripts/live_inference_agcn.py` (now runs
MediaPipe Hands alongside Pose) before trusting this over v3's test-set
numbers -- the whole point of this version was fixing a live-only failure
that the locked test set didn't expose. If it holds up live, it's the
deployment candidate; `src/inference.py`'s `ActionSegmenter` now buffers
hand landmarks via its optional `aux_frame` param for exactly this.